# Task 1: Fine-Tuning Qwen2.5-0.5B with LoRA for Structured Intent Extraction
### tndrr AI/ML Internship Technical Assessment
**Author:** Harsh Maurya | **Repository:** [applied-llm-systems](https://github.com/Advik-harsha/applied-llm-systems)

This notebook demonstrates fine-tuning `Qwen/Qwen2.5-0.5B-Instruct` using Low-Rank Adaptation (LoRA) on consumer/Colab T4 GPUs. It trains the model on custom customer support queries and converts them to deterministic, schema-validated JSON outputs.

In [ ]:
# 1. Install dependencies
!pip install -q transformers peft trl datasets accelerate bitsandbytes

In [ ]:
import torch
import json
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
from trl import SFTTrainer

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Load Model & Tokenizer
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True
)

# 3. Configure LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# 4. Training configuration & SFTTrainer
# Note: Load train.jsonl and test.jsonl from repository or generate locally
training_args = TrainingArguments(
    output_dir="./lora_output",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    num_train_epochs=3,
    logging_steps=10,
    fp16=torch.cuda.is_available(),
    report_to="none"
)
print("Training arguments configured successfully.")